# Justice Domain Fairness Audit: COMPAS Recidivism Dataset

## Overview

This notebook demonstrates a complete fairness audit using the FDK Toolkit on the COMPAS recidivism risk assessment dataset.

**Source:** FDK Manuscript, Chapter 10.3, pp. 150–152; Chapter 36, pp. 323–329

**Dataset:** COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) – recidivism risk scores assigned to criminal defendants

**Domain:** Justice – criminal justice, recidivism, bail decisions

**Objective:** Identify and measure demographic disparities in recidivism risk predictions

## Step 1: Import Required Libraries

In [ ]:
# FDK Toolkit imports
from fdk import FairnessAudit
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)

## Step 2: Load the COMPAS Dataset

**Source:** Chapter 36, p. 324 – n = 7,214 records

In [ ]:
# Load COMPAS data
# In practice, this would be: df = pd.read_csv('compas_data.csv')

# For demonstration, we show the expected structure
print("Expected columns: ")
print("- protected_attributes: race, gender, age")
print("- outcome: two_year_recid (actual recidivism)")
print("- prediction: decile_score (risk score 1-10)")

## Step 3: Define the Justice Outcome

**Source:** Chapter 10.3, p. 150

> *"The auditing team begins by selecting a clear outcome for evaluation: whether the algorithm predicted recidivism (re-offending within two years)."*

In [ ]:
# Define target variable and protected attributes
target_column = 'two_year_recid'  # 0 = no recidivism, 1 = recidivism
protected_attributes = ['race', 'gender', 'age']
prediction_column = 'decile_score'  # Risk score (higher = higher predicted risk)

## Step 4: Run the FDK Fairness Audit

**Source:** Chapter 10.3, p. 151

> *"Using FDK™, the COMPAS dataset is analyzed across multiple fairness dimensions. Protected groups are defined based on available attributes (race, age, gender, and socioeconomic status proxies), and the system automatically evaluates disparities across these groups."*

In [ ]:
# Initialize the fairness audit for Justice domain
audit = FairnessAudit(domain='justice')

# Run the audit
# results = audit.run(df, target=target_column, protected=protected_attributes, predictions=prediction_column)

# For demonstration, we show expected results from the manuscript

## Step 5: Review Key Fairness Metrics

**Source:** Chapter 10.3, p. 151 – Table 10.1

In [ ]:
# Expected results from COMPAS fairness audit (Source: Chapter 10.3, p. 151)

results = {
    'composite_bias_score': 0.1165,
    'disparate_impact_ratio': 0.5625,
    'statistical_parity_difference': 0.2674,
    'false_discovery_rate_difference': 0.1288,
    'counterfactual_fairness': 0.7326,
    'group_selection_rates': {
        'Caucasian': 0.222,
        'African_American': 0.088,
        'Hispanic': 0.120,
        'Native_American': 0.100
    },
    'accuracy_by_group': {
        'Caucasian': 0.777,
        'African_American': 0.583,
        'Hispanic': 0.650,
        'Native_American': 0.600
    }
}

# Display results
print("=" * 60)
print("FAIRNESS AUDIT RESULTS - COMPAS RECIDIVISM")
print("=" * 60)
print(f"Composite Bias Score: {results['composite_bias_score']:.4f}")
print(f"Disparate Impact Ratio: {results['disparate_impact_ratio']:.4f}")
print(f"Statistical Parity Difference: {results['statistical_parity_difference']:.4f}")
print(f"False Discovery Rate Difference: {results['false_discovery_rate_difference']:.4f}")
print(f"Counterfactual Fairness: {results['counterfactual_fairness']:.4f}")

## Step 6: Interpret the Results

**Source:** Chapter 10.3, p. 151–152

> *"These results indicate that the recidivism risk algorithm does not treat all demographic groups equally. In particular: The disparate impact ratio (0.5625) suggests that one group is predicted to re-offend at roughly half the rate of another, falling well below the commonly accepted 0.8 threshold for fairness. The statistical parity difference (0.2674) confirms unequal prediction rates across groups. The false discovery rate difference (0.1288) shows that when the algorithm predicts re-offending, it is wrong more often for certain groups than others."*

In [ ]:
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

if results['disparate_impact_ratio'] < 0.8:
    print("⚠️ DISPARATE IMPACT RATIO below 0.8 threshold - ACTION REQUIRED")
    print(f"   Value: {results['disparate_impact_ratio']:.4f} (threshold: 0.8)")

if results['composite_bias_score'] > 0.1:
    print("⚠️ COMPOSITE BIAS SCORE above 0.1 - HIGH BIAS detected")
    print(f"   Value: {results['composite_bias_score']:.4f} (HIGH_BIAS range: >0.10)")

print("\nFrom a practical perspective:")
print("- Defendants from different demographic backgrounds may face systematically")
print("  different risk classifications, even when their underlying criminal")
print("  histories are comparable.")
print("- The algorithm is wrong more often for certain groups than others.")
print("- Review and potential remediation are required.")

## Step 7: Governance Escalation

**Source:** Chapter 8.6, p. 126–128; Chapter 36, p. 325

In [ ]:
def determine_governance_action(bias_score, disparate_impact):
    """
    Determine governance action based on fairness metrics.
    Source: Chapter 8.6, p. 128 - Threshold mapping table
    """
    if disparate_impact < 0.8 or bias_score > 0.10:
        return "CRITICAL - Escalate to Technical Review Board. Suspend deployment pending review."
    elif disparate_impact < 0.9 or bias_score > 0.05:
        return "CONCERNING - Operational Fairness Committee review required. Re-tune or recalibrate."
    else:
        return "ACCEPTABLE - Continue monitoring. No intervention required."

action = determine_governance_action(results['composite_bias_score'], results['disparate_impact_ratio'])
print(f"Governance Action: {action}")

## Step 8: Generate Evidence Pack

**Source:** Chapter 8.3, p. 121 – *"An evidence pack combines: what was tested, how it was tested, what the results were, what action was taken"*

In [ ]:
import json
from datetime import datetime

evidence_pack = {
    'audit_id': f"COMPAS_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    'domain': 'justice',
    'dataset': 'COMPAS recidivism',
    'sample_size': 7214,
    'protected_attributes': protected_attributes,
    'target_variable': target_column,
    'results': results,
    'governance_action': action,
    'methodological_assumptions': [
        "Independence of observations assumed",
        "Groups with n<30 flagged for caution",
        "Confidence intervals reported where data permits"
    ],
    'limitations': [
        "Disparate impact ratio does not account for legitimate risk differences",
        "Composite bias score is a weighted aggregate, not a definitive verdict",
        "Fairness metrics may conflict; single-metric conclusions are insufficient"
    ],
    'timestamp': datetime.now().isoformat()
}

# Save evidence pack
with open(f"{evidence_pack['audit_id']}_evidence_pack.json", 'w') as f:
    json.dump(evidence_pack, f, indent=2)

print(f"Evidence pack saved: {evidence_pack['audit_id']}_evidence_pack.json")
print("\nThis evidence pack is suitable for:")
print("- Internal governance review")
print("- Regulatory submission")
print("- External audit")

## Summary

**Source:** Chapter 10.3, p. 152

> *"This worked example shows that fairness issues in justice algorithms can be identified before any corrective action is taken. By analyzing real recidivism risk predictions and presenting clear metrics, FDK™ enables courts, auditors, and policymakers to detect and understand disparities early, supporting more accountable and transparent algorithmic justice."*

**Key Takeaways:**
1. The COMPAS dataset exhibits measurable disparities across racial groups
2. Disparate impact ratio (0.5625) falls well below the 0.8 threshold
3. Composite bias score (0.1165) indicates HIGH_BIAS
4. Governance escalation is required (CRITICAL level)
5. Evidence pack created for audit trail and regulatory submission

**Next Steps:**
- Escalate to Technical Review Board
- Consider BiasClean remediation (see Chapter 36)
- Re-audit after any model or data changes